In [ ]:
%load_ext autoreload
%autoreload 2
%cd /opt/tiger/samantha

In [ ]:
# from recipes.research.audio_codec import AudioCodec, AudioCodecConfig

# config = AudioCodecConfig(
#     sample_rate=44100,
# )
# audio_codec = AudioCodec(config)

# Evaluate

In [ ]:
from recipes.research.audio_codec.scripts.compile import get_audio_codec
# audio_codec = get_audio_codec("4cec122") # 24khz, 25hz, 32d, stereo
# audio_codec = get_audio_codec("c6ba872") # 38khz, 20hz, 64d, mono (less steps)
# audio_codec = get_audio_codec("66618b7") # 38khz, 20hz, 64d, mono
# audio_codec = get_audio_codec("32aee0d") # 38khz, 20hz, 32d, mono
# audio_codec = get_audio_codec("cffbe3b") # 38khz, 20hz, 32d, stereo
# audio_codec = get_audio_codec("5021925") # 38khz, 20hz, 16d, stereo

# audio_codec = get_audio_codec("d29a86d") # 44.1kHz, 25hz, 64d, stereo
# audio_codec = get_audio_codec("8bfa192") # 44.1kHz, 25hz, 64d, stereo
# audio_codec = get_audio_codec("0f25752") # 44.1kHz, 21.5Hz, 64d, stereo

# audio_codec = get_audio_codec("310f946") # 44.1kHz, 25Hz, 64d, stereo, 1e-4
# audio_codec = get_audio_codec("b0718b8") # 44.1kHz, 25Hz, 64d, stereo, 1e-4

# audio_codec = get_audio_codec("b0718b8") # 44.1kHz, 25Hz, 64d, stereo, 1e-4
# audio_codec = get_audio_codec("a85c2b3")
# audio_codec = get_audio_codec("a153472")
# audio_codec = get_audio_codec("6df9d40")
# audio_codec = get_audio_codec("f81b3fa")
# audio_codec = get_audio_codec("55fbc56")

# audio_codec = get_audio_codec("2c8072b")

# audio_codec = get_audio_codec("b95ba32")

# audio_codec = get_audio_codec("0704ec3") # 25h
# audio_codec = get_audio_codec("16d7ad6") # 50h

# audio_codec = get_audio_codec("bab4970")

audio_codec = get_audio_codec("914dbd2")

audio_codec = audio_codec.to("cuda")

In [ ]:
import os
import torch.nn.functional as F
import torch
import torchaudio
from samantha.data.utils import read_audio
from IPython.display import Audio, display
from tqdm import tqdm
from samantha.transforms.audio import batch_plot_spectrogram
from samantha.data.av_audio import audio_write

fps = [
    "/mnt/bn/janne-research-xl/assets/music/bob dylan/02 - Girl from the North Country.mp3",
    "/mnt/bn/janne-research-xl/assets/music/bob dylan/03 - Masters of War.mp3",
    "/mnt/bn/janne-research-xl/assets/music/01 - The Times They Are A-Changin'.mp3",
    "/mnt/bn/janne-research-xl/assets/music/10 No Surprises.mp3",
    "/mnt/bn/janne-research-xl/assets/music/french/11 - L'amour.mp3",
    "/mnt/bn/janne-research-xl/assets/music/amy_winehouse/13 - Rehab.mp3",
    "/mnt/bn/janne-research-xl/assets/music/amy_winehouse/06 - Back To Black.mp3",
    "/mnt/bn/janne-research-xl/assets/music/Paul Kalkbrenner - Sky and Sand (Official Music Video).mp3",
    "/mnt/bn/janne-research-xl/assets/music/11 - Billie Eilish - What Was I Made For.mp3",
    "/mnt/bn/janne-research-xl/assets/music/16 - Billie Eilish - No Time To Die.mp3",
    "/mnt/bn/janne-research-xl/assets/music/Fred again.., Lil Yachty & Overmono - stayinit.mp3",
    "/mnt/bn/janne-research-xl/assets/music/01-02 Can You Hear The Music.mp3",
    "/mnt/bn/janne-research-xl/assets/music/24. Fellowship.mp3",
    "/mnt/bn/janne-research-xl/assets/music/instrumental/42 Rey's Theme.mp3",
    "/mnt/bn/janne-research-xl/assets/music/instrumental/26 - Back to the Future - End Credits.mp3",
    "/mnt/bn/janne-research-xl/assets/music/instrumental/Hans Zimmer - Time.mp3",
    "/mnt/bn/janne-research-xl/assets/music/instrumental/02 - Chevaliers De Sangreal (From The Da Vinci Code Original Motion Picture Soundtrack).mp3",
]

BASE_DIR = os.path.join("/mnt/bn/janne-research-xl/demo", audio_codec.commit_hash, audio_codec.commit_step)
os.makedirs(BASE_DIR, exist_ok=True)
print(BASE_DIR)
for fp in tqdm(fps):
    fn = os.path.basename(fp)
    audio, sr = read_audio(
        fp,
        audio_codec.sample_rate,
        normalize_loudness=True
    )
    audio = audio.to("cuda")
    # audio = audio.mean(dim=1, keepdim=True)
    
    with torch.no_grad():
        # hop_length = audio_codec.hop_length
        # audio = F.pad(audio, (0, hop_length - (audio.shape[2] % hop_length)))
        z = audio_codec.get_z(audio, sr)
        print(z.shape, z.mean(), z.std(), audio_codec.frame_rate)
        
        batch_plot_spectrogram(z.cpu(), plot_log=False)
        rec_audio = audio_codec.decode_z(z)
        
        # mp3_bytes = audio_write(rec_audio[0].cpu(), audio_codec.sample_rate, format="mp3")
        
        # display(Audio(mp3_bytes))
    
    torchaudio.save(os.path.join(BASE_DIR, f"{fn}_gen.flac"), rec_audio[0].cpu(), audio_codec.sample_rate)
    torchaudio.save(os.path.join(BASE_DIR, f"{fn}_real.flac"), audio[0].cpu(), audio_codec.sample_rate)

# Zoo

In [ ]:
from recipes.research.audio_codec.zoo import AudioCodec, AudioCodec_66618b7_64l, AudioCodec_d29a86d_64l, AudioCodec_0f25752_64l, AudioCodec_310f946_64l, AudioCodec_f81b3fa_64l, AudioCodec_7c355ea_64l

# audio_codec = AudioCodec_66618b7_64l()
# audio_codec = AudioCodec_d29a86d_64l()
# audio_codec = AudioCodec_0f25752_64l()
# audio_codec = AudioCodec_310f946_64l()

# audio_codec = AudioCodec()

# audio_codec = AudioCodec_f81b3fa_64l()

audio_codec = AudioCodec_7c355ea_64l()
audio_codec = audio_codec.to("cuda")

print(audio_codec.sample_rate)
print(audio_codec.frame_rate)
print(audio_codec.hop_length)
print(audio_codec.latent_dim)
print(audio_codec.n_channels)
print(audio_codec.mean)
print(audio_codec.std)
print(audio_codec.vae_beta)

audio_codec.commit_hash = "7c355ea"
audio_codec.commit_step = "S"

In [ ]:
import os
import torch.nn.functional as F
import torch
import matplotlib.pyplot as plt
from samantha.data.utils import read_audio
from IPython.display import Audio, display
from samantha.transforms.audio import plot_spectrogram
from samantha.transforms.audio import batch_plot_spectrogram, batch_plot_amplitude_histogram
from recipes.research.audio_codec.models.uac import UACResult

from samantha.transforms.audio import MelSpectrogram
from recipes.research.mel_codec.models.umc_mki import MelSpectrogramFeatures


# BASE_DIR = os.path.join("/mnt/bn/janne-research-xl/demo", audio_codec.commit_hash, audio_codec.commit_step)
# os.makedirs(BASE_DIR, exist_ok=True)
# print(BASE_DIR)

# fp = "/mnt/bn/janne-research-xl/assets/music/10 No Surprises.mp3"
# fp = "/mnt/bn/janne-research-xl/assets/music/instrumental/42 Rey's Theme.mp3"
# fp = "/mnt/bn/janne-research-xl/assets/music/instrumental/02 - Chevaliers De Sangreal (From The Da Vinci Code Original Motion Picture Soundtrack).mp3"
# fp = "/mnt/bn/janne-research-xl/assets/music/amy_winehouse/06 - Back To Black.mp3"
fp = "/mnt/bn/janne-research-xl/assets/music/instrumental/01-02 Can You Hear The Music.mp3"
# fp = "/mnt/bn/janne-research-xl/assets/music/instrumental/26 - Back to the Future - End Credits.mp3"

audio, sr = read_audio(
    fp,
    audio_codec.sample_rate,
    normalize_loudness=True
)
audio = audio.to("cuda")

mel_transform = MelSpectrogramFeatures(
    sr,
    n_mels=160,
    n_fft=4096,
    hop_length=441,
    f_max=20000,
).to("cuda")

train_sec = 0.76
shift_sec = 0
cur_sec = 30
use_window = False
audio = audio[..., sr*shift_sec:sr*shift_sec + sr*cur_sec]


fn = os.path.basename(fp)

with torch.no_grad():
    print(audio.shape)
    audio = F.pad(audio, (0, 2048 - (audio.shape[2] % 2048)))
    print(audio.shape)
    z = audio_codec.get_z(audio, sr)
    x = audio_codec.decode_z(z)
        
    
    result = UACResult(
        audio=x, # keep consistent
        latents=z,
        commitment_loss=None,
        codebook_loss=None,
    )
    
    batch_plot_spectrogram(result.latents.cpu(), plot_log=False, figsize=(20, 10))
    plt.show()
    
        
    fig, ax = plt.subplots(3, 3, figsize=(25, 15))
    plt.suptitle(f"{fn} - ConvCodec | train_sec: {train_sec}, use_window: {use_window}, extract_sec: {cur_sec}")
    ax = ax.flatten()

    # plot energy distribution:
    batch_plot_amplitude_histogram(audio, bins=100, ax=ax[0])
    batch_plot_amplitude_histogram(result.latents, bins=100, ax=ax[1])
    batch_plot_amplitude_histogram(audio, bins=100, ax=ax[2], imshow_kwargs={"alpha": 0.5, "label": "y"})
    batch_plot_amplitude_histogram(result.audio, bins=100, ax=ax[2], imshow_kwargs={"alpha": 0.5, "label": "rec"})

    
    mel = mel_transform(audio.mean(dim=1))
    batch_plot_spectrogram(mel, plot_log=False, ax=ax[3], imshow_kwargs={})

    
    batch_plot_spectrogram(result.latents.cpu(), plot_log=False, ax=ax[4], imshow_kwargs={})
    
    rec_mel = mel_transform(result.audio.mean(dim=1))
    batch_plot_spectrogram(rec_mel, plot_log=False, ax=ax[5], imshow_kwargs={})
    
    print(mel.shape, rec_mel.shape)
    
    batch_plot_spectrogram((mel - rec_mel).abs().cpu(), plot_log=False, ax=ax[6])
    
    # plt.savefig(os.path.join(BASE_DIR, f"{fn}-{cur_sec}-plots.png"))
    plt.show()
    plt.close()

display(Audio(x[0].cpu(), rate=audio_codec.sample_rate, normalize=True))

In [ ]:
import matplotlib.pyplot as plt
from samantha.transforms.audio import Spectrogram, batch_plot_spectrogram, generate_sine, generate_silence, plot_spectrogram


sample_rate = 24000
freq = 240
duration = 5
amplitude = 1.0
n_fft = 4096
wave = generate_sine(sample_rate, duration, freq, amplitude, device="cuda")

wave = wave[None]

s = Spectrogram(
    n_fft=64,
    win_length=64,
    hop_length=sample_rate//25,
    window_fn=torch.signal.windows.kaiser,
)
s = s.to("cuda")

spec, _ = s(wave)
print(spec.shape)

spec = spec[:, :, :-1]
plot_spectrogram(spec[0][0].cpu(), plot_log=False)

In [ ]:
import matplotlib.pyplot as plt
from samantha.transforms.audio import Spectrogram, batch_plot_spectrogram, generate_sine, generate_silence, plot_spectrogram

# frame rate = 25Hz
sample_rate = audio_codec.sample_rate

n_fft = audio_codec.sample_rate // 8
win_length = n_fft // 2
s = Spectrogram(
    n_fft=n_fft,
    win_length=win_length,
    hop_length=win_length//4,
)
s = s.to("cuda")


# fig, axs = plt.subplots(1, 10, figsize=(50, 50), dpi=300)
# axs = axs.flatten()

for freq, ax in zip(range(7500, 7600, 5), axs):
# for freq in range(1500, 1700, 20):
    amplitude = 1.0
    duration = 5
    wave = generate_sine(sample_rate, duration, freq, amplitude, device="cuda")
    
    wave = wave[None]
    wave = wave.repeat(1, 2, 1)
    
    with torch.no_grad():
        z = audio_codec.get_z(wave, sample_rate)
        # print(z.shape, z.mean(), z.std(), audio_codec.frame_rate)
        rec_audio = audio_codec.decode_z(z)
        
    break
        
#     fig, ax = plt.subplots(1, 1, figsize=(10,10))
#     plot_spectrogram(z[0].cpu(), plot_log=False, ax=ax, title=f"sine freq: {freq}Hz")
#     plt.show()


    # plot_spectrogram(z[0].cpu(), plot_log=False, ax=ax, title=f"sine freq: {freq}Hz")
    # display(Audio(rec_audio[0].cpu(), rate=audio_codec.sample_rate, normalize=True))
    
    # spec, _ = s(rec_audio)
    # plot_spectrogram(spec[0, 0].cpu(), plot_log=False, title=f"sine freq: {freq}Hz")
    # plt.show()

#     # plot spec
#     n_fft = audio_codec.frame_rate // 3
#     win_length = n_fft // 2
#     s = Spectrogram(
#         n_fft=n_fft,
#         win_length=win_length,
#         hop_length=win_length//4,
#     )
#     s = s.to("cuda")

#     specs = []
#     for c in range(audio_codec.latent_dim):
#         spec, _ = s(z[0, c])
#         spec = spec[..., :-1]
#         spec = spec.mean(dim=0, keepdim=True)
#         specs.append(spec)

#     specs = torch.cat(specs, dim=0)
    
#     plot_spectrogram(specs.cpu(), plot_log=False, ax=ax, title=f"sine freq: {freq}Hz")

# plt.tight_layout()
# plt.savefig("z.pdf")
# # plt.savefig("spec.pdf")

In [ ]:
from samantha.data.utils import read_audio
from IPython.display import Audio, display
from samantha.transforms.audio import batch_plot_spectrogram, batch_plot_amplitude_histogram

# fp = "/mnt/bn/janne-research-xl/assets/music/instrumental/42 Rey's Theme.mp3"
fp = "/mnt/bn/janne-research-xl/assets/music/instrumental/02 - Chevaliers De Sangreal (From The Da Vinci Code Original Motion Picture Soundtrack).mp3"
# fp = "/mnt/bn/janne-research-xl/assets/music/amy_winehouse/06 - Back To Black.mp3"
# fp = "/mnt/bn/janne-research-xl/assets/music/instrumental/01-02 Can You Hear The Music.mp3"
# fp = "/mnt/bn/janne-research-xl/assets/music/instrumental/26 - Back to the Future - End Credits.mp3"

In [ ]:
audio, sr = read_audio(fp, 44100, normalize_loudness=False)

audio = audio.to("cuda")

# Compute statistics

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /opt/tiger/samantha

# from samantha.data.audio.webdataset import AudioWebDataset
from samantha.data.audio.dataset import AudioFolderDataModule, AudioFolderDataset

duration = 30.0
# billboard = AudioWebDataset(
#     url2index="hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data/js/data/music/shards/billboard_hot_200-v2_normalised_-16LUFS/*/url2index.txt",
#     sample_rate=audio_codec.sample_rate,
#     channels=2,
#     pad=True,
#     segment_duration=duration,
#     resampled=True,
#     shardshuffle=True,
#     shuffle_buffer_size=10,
# )

# jamendo = AudioWebDataset(
#     url2index="hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data/js/data/music/shards/jamendo_normalised_-16LUFS_mp3/*/url2index.txt",
#     sample_rate=audio_codec.sample_rate,
#     channels=2,
#     pad=True,
#     segment_duration=duration,
#     resampled=True,
#     shardshuffle=True,
#     shuffle_buffer_size=10,
# )


# instrumental_hq = AudioFolderDataset(
#   root="data/instrumental_hq",
#   pad=True,
#   duration=duration,
#   sample_rate=audio_codec.sample_rate,
#   shuffle=True,
#   num_channels=2,
#   num_workers=16,
# )

from recipes.research.dataset.collection import EveryNoiseParquetDataset, ShutterStockParquetDataset
# everynoise = EveryNoiseParquetDataset(
#     sample_rate=audio_codec.sample_rate,
#     channels=2,
#     segment_duration=duration,
#     resampled=True,
#     shardshuffle=True,
# )

shutterstock = ShutterStockParquetDataset(
    sample_rate=audio_codec.sample_rate,
    channels=2,
    segment_duration=duration,
    resampled=True,
    shardshuffle=True,
    crop_from_start=True
)

In [ ]:
from samantha.data.audio.dataset import AudioFolderDataModule, AudioFolderDataset

batch_size = 4
datamodule = AudioFolderDataModule(
    [shutterstock],
    [],
    [],
    weights=None,
    batch_size=batch_size,
    shuffle=None,
    num_workers=16
)
dataloader = datamodule.train_dataloader()

In [ ]:
import torch
n_batches = 1000

from tqdm import tqdm

zs = []
with torch.no_grad():
    for idx, batch in enumerate(tqdm(dataloader, total=n_batches)):
        if idx == n_batches:
            break
        batch.audio = batch.audio.to("cuda")
        
        if audio_codec.n_channels == 1:
            batch.audio = batch.audio.mean(dim=1, keepdim=True)
    
        z = audio_codec.get_z(batch.audio, audio_codec.sample_rate)
        zs.append(z)
        
        if idx % 100 == 0:
            z_tmp = torch.cat(zs, dim=0)
            print(z_tmp.mean(), z_tmp.var(), z_tmp.std())
zs = torch.cat(zs, dim=0)

In [ ]:
print(zs.mean().item(), zs.std().item())